In [ ]:
!which python

/home/sa1/Documents/gitReps/pythonQuickAndDirty/venv/bin/python


In [5]:
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src" / "data" / "SmartEM" / "Philips").exists():
    repo_root = repo_root.parents[2]

data_dir = repo_root / "src" / "data" / "SmartEM" / "Philips"

input_data = pd.read_csv(data_dir / "input.csv")
output_data = pd.read_csv(data_dir / "output.csv")

input_data, output_data

(      input_1  input_2  input_3  input_4  input_5 input_6  input_7  \
 0         C01       79       83       57       82     189      233   
 1         C01       79       83       57       82     189      233   
 2         C01       79       83       57       82     189      233   
 3         C01       79       83       57       82     189      233   
 4         C01       79       83       57       82     189      233   
 ...       ...      ...      ...      ...      ...     ...      ...   
 41327     C05       88       55       43       82     234      207   
 41328     C05       88       55       43       82     234      207   
 41329     C05       88       55       43       82     234      207   
 41330     C05       88       55       43       82     234      207   
 41331     C05       88       55       43       82     234      207   
 
                input_8      input_9 input_10  ...  input_13   input_14  \
 0          290-360-260  366-388-364   V-type  ...         9  62.311333

In [10]:
input_data.nunique()
input_data.describe()

,input_2,input_3,input_4,input_5,input_7,input_11,input_13,input_14,input_15,input_16,input_17,input_18,input_19,input_20,input_21,input_22
count,41332.000000,41332.000000,41332.000000,41332.000000,41332.000000,41332.000000,41332.0,41332.000000,41332.0,41332.0,41332.0,41332.000000,41332.000000,41332.000000,41332.000000,41332.000000
mean,82.985677,68.886771,54.914328,79.402836,231.041034,14.529662,9.0,47.016870,150.0,2100.0,250.0,-11.550573,-0.093421,516.181095,179.801444,19.930309
std,5.914212,10.898320,26.421992,47.105885,7.342805,1.612722,0.0,14.621300,0.0,0.0,0.0,1.367322,5.805898,463.575652,103.925452,5.172689
min,73.000000,55.000000,0.000000,50.000000,207.000000,9.000000,9.0,20.073097,150.0,2100.0,250.0,-12.999000,-9.900000,50.019867,0.002608,-2.317988
25%,79.000000,55.000000,43.000000,50.000000,233.000000,15.000000,9.0,34.638755,150.0,2100.0,250.0,-12.770000,-5.356000,93.843252,89.182823,16.197775
50%,81.000000,75.000000,64.000000,50.000000,233.000000,15.000000,9.0,48.559010,150.0,2100.0,250.0,-11.987000,-0.162000,215.507617,179.434411,19.928539
75%,91.000000,75.000000,68.000000,82.000000,233.000000,15.000000,9.0,59.891366,150.0,2100.0,250.0,-10.527000,5.031000,972.810968,269.346158,23.638489
max,91.000000,83.000000,79.000000,180.000000,236.000000,15.000000,9.0,69.922459,150.0,2100.0,250.0,-8.426000,9.900000,1657.842136,359.977664,41.296990


In [2]:
import re

odd_columns = ["input_6", "input_8", "input_9"]
expected_dims = {"input_6": 2, "input_8": 4, "input_9": 3}
token_pattern = re.compile(r"^\d+(?:-\d+)*$")

input_processed = input_data.copy()


def parse_tokens(value):
    text = str(value).strip()
    if not token_pattern.fullmatch(text):
        raise ValueError(f"Unexpected token format in odd column: {value!r}")
    return [int(part) for part in text.split("-")]


def normalize_tokens(tokens, width):
    if len(tokens) > width:
        raise ValueError(f"Token length {len(tokens)} exceeds expected width {width}")
    padded = tokens + [0] * (width - len(tokens))
    return "-".join(str(v) for v in padded)


observed_dims = {}
for col in odd_columns:
    parsed = input_data[col].astype("string").map(parse_tokens)
    observed_dims[col] = int(parsed.map(len).max())
    input_processed[col] = parsed.map(lambda t: normalize_tokens(t, expected_dims[col]))

assert observed_dims == expected_dims, f"Unexpected max dims: {observed_dims}"

# Spot-check requested examples.
assert normalize_tokens(parse_tokens("189"), expected_dims["input_6"]) == "189-0"
assert normalize_tokens(parse_tokens("340-260"), expected_dims["input_8"]) == "340-260-0-0"
assert normalize_tokens(parse_tokens("651-360-455-312"), expected_dims["input_8"]) == "651-360-455-312"
assert normalize_tokens(parse_tokens("0"), expected_dims["input_8"]) == "0-0-0-0"

# Keep dataset unchanged except offending columns.
untouched_columns = [c for c in input_data.columns if c not in odd_columns]
for col in untouched_columns:
    assert input_processed[col].equals(input_data[col]), f"Unexpected change in {col}"

assert list(input_processed.columns) == list(input_data.columns)
assert len(input_processed) == len(input_data)

# Verify fixed widths in-place.
for col, width in expected_dims.items():
    lens = input_processed[col].astype("string").str.split("-").str.len()
    assert lens.eq(width).all(), f"Width mismatch in {col}"

processing_summary = {
    "shape": input_processed.shape,
    "column_count": len(input_processed.columns),
    "changed_columns": odd_columns,
    "observed_dims": observed_dims,
    "samples": {
        col: input_processed[col].head(3).tolist() for col in odd_columns
    },
}

processing_summary



{'shape': (41332, 22),
 'column_count': 22,
 'changed_columns': ['input_6', 'input_8', 'input_9'],
 'observed_dims': {'input_6': 2, 'input_8': 4, 'input_9': 3},
 'samples': {'input_6': ['189-0', '189-0', '189-0'],
  'input_8': ['290-360-260-0', '290-360-260-0', '290-360-260-0'],
  'input_9': ['366-388-364', '366-388-364', '366-388-364']}}

In [3]:
input_processed_path = data_dir / "input_processed.csv"
input_processed.to_csv(input_processed_path, index=False)

{
    "saved_to": str(input_processed_path),
    "shape": input_processed.shape,
    "column_count": len(input_processed.columns),
    "changed_columns": odd_columns,
}



{'saved_to': '/home/sa1/Documents/gitReps/pythonQuickAndDirty/src/data/SmartEM/Philips/input_processed.csv',
 'shape': (41332, 22),
 'column_count': 22,
 'changed_columns': ['input_6', 'input_8', 'input_9']}